In [ ]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)    # Muestra todas las filas (para muchas columnas)
pd.set_option('display.max_columns', None) # Muestra todas las columnas
pd.set_option('display.width', None)       # Ajusta el ancho
pd.set_option('display.max_colwidth', None)# Muestra todo el contenido de cada celda

In [ ]:
drivers_c = pd.read_csv('csv_limpios/drivers_c.csv')
drivers_updated_c = pd.read_csv('csv_limpios/drivers_updated_c.csv')

In [ ]:
drivers_c

In [ ]:
drivers_updated_c

### Elimino columna CODE en drivers_c y drivers_updated_c porque no la voy a utilizar

In [ ]:
drivers_updated_c.drop(columns=['Code'], inplace=True)
drivers_c.drop(columns=['code'], inplace=True)

In [ ]:
# Elimino la columna de Drivers_ID de drivers_updated_c
drivers_updated_c.drop(columns=['Driver_ID'], inplace=True)

### Uno las columnas 'forename' y 'surname' de drivers_c en la columna creada 'Driver'

In [ ]:
# Creo columna 'Driver' en drivers_c juntando 'forename' y 'surname'
drivers_c['Driver'] = drivers_c['forename'] + ' ' + drivers_c['surname']

In [ ]:
# Posiciono la columna 'Driver' al inicio del DataFrame
drivers_c = drivers_c[['Driver'] + [col for col in drivers_c.columns if col != 'Driver']]

In [ ]:
# Elimino las columnas 'forename', 'surname' y 'driverRef' de drivers_c
drivers_c.drop(columns=['forename', 'surname', 'driverRef'], inplace=True)

### Relleno los null con los datos correctos de cada piloto en la columna Car

In [ ]:
# Muestro solo las filas donde hay un null en la columna 'Car' dentro de drivers_updated_c
drivers_updated_c[drivers_updated_c['Car'].isnull()]

In [ ]:
# Creo lista para rellenar datos de la columna 'Car' de drivers_updated_c donde hay un null

mapping_id = {
    (17, 1950): 'Ferrari',
    (18, 1950): 'Talbot-Lago',
    (22, 1950): 'Talbot-Lago',
    (36, 1951): 'Talbot-Lago',
    (54, 1952): 'Cooper Bristol',
    (310, 1963): 'Reg Parnell Racing',
    (315, 1963): 'Reg Parnell Racing',
    (447, 1969): 'Lotus Ford',
    (479, 1971): 'STP March Racing Team',
    (493, 1971): 'McLaren Ford',
    (494, 1971): 'Frank Williams Racing Cars/Williams',
}

def asignar_car_id(row):
    key = (row['Driver_ID'], row['year'])
    return mapping_id.get(key, row['Car'])

drivers_updated_c['Car'] = drivers_updated_c.apply(asignar_car_id, axis=1)


--------------------------------------------------

### Pilotos Unicos 'drivers_updated_c'

In [ ]:
pilotos_unicos = drivers_updated_c['Driver'].unique().tolist()
pilotos_unicos.sort()  # ordena alfabéticamente

for piloto in pilotos_unicos:
    print(piloto)

In [ ]:
# Sumar los valores totales dentro de la variable pilotos_unicos
total_pilotos = len(pilotos_unicos)
print(f'Total de pilotos únicos: {total_pilotos}')

### Pilotos Unicos 'drivers_c'

In [ ]:
pilotos_unicos2 = drivers_c['Driver'].unique().tolist()
pilotos_unicos2.sort()  # ordena alfabéticamente

for piloto in pilotos_unicos2:
    print(piloto)

In [ ]:
# Sumar los valores totales dentro de la variable pilotos_unicos
total_pilotos2 = len(pilotos_unicos2)
print(f'Total de pilotos únicos: {total_pilotos2}')

------------------------------------------------

### Escuderias totales

In [ ]:
# Muestro los valores unicos de la columna 'Car' dentro de drivers_updated_c
drivers_updated_c['Car'].drop_duplicates()

-----------------------------------------

//////////////////////////////////////////////////

---------------------------------------

### Guardo cambios de los DataFrames

In [ ]:
# Creo codigo para guardar los DataFrames actualizados
drivers_c.to_csv('csv_limpios/drivers_data.csv', index=False)
drivers_updated_c.to_csv('csv_limpios/drivers_GP.csv', index=False)

In [ ]:
# Creo codigo para cargar los DataFrames actualizados
drivers_data = pd.read_csv('csv_limpios/drivers_data.csv')
drivers_GP = pd.read_csv('csv_limpios/drivers_GP.csv')

In [ ]:
drivers_data

In [ ]:
drivers_GP

### Sincronizar ambos df con 'Driver_ID' y corregir incompatibilidades entre columnas 'Driver'

In [ ]:
# Creo una columna llamada 'Driver_ID' en drivers_c que funcione como clave primaria y se posicione al principio del DataFrame
drivers_data.insert(0, 'Driver_ID', drivers_data.index + 1)

In [ ]:
# Corroborar que ambas columnas 'Driver' en ambos DataFrames solo tengan un espacio entre palabras
drivers_data['Driver'] = drivers_data['Driver'].str.replace(r'\s+', ' ', regex=True)
drivers_GP['Driver'] = drivers_GP['Driver'].str.replace(r'\s+', ' ', regex=True)

In [ ]:
# Corroborar que la columna de 'Driver' solo tenga un nombre y un apellido, sin espacios ni palabras adicionales
drivers_data['Driver'] = drivers_data['Driver'].str.strip()
drivers_GP['Driver'] = drivers_GP['Driver'].str.strip()

In [ ]:
# Corroborar que ambas columnas 'Driver' no tengan caracteres especiales y si tienen que se reemplacen por la letra sin el caracter especial, incluir mayusculas y minúsculas
def limpiar_caracteres_especiales(nombre):
    caracteres_especiales = {
        'ñ': 'n', 'Ñ': 'N', 'á': 'a', 'Á': 'A', 'é': 'e', 'É': 'E',
        'í': 'i', 'Í': 'I', 'ó': 'o', 'Ó': 'O', 'ú': 'u', 'Ú': 'U',
        '-': '', '_': '', '.': '', ',': '', 'è': 'e', 'ô': 'o', 
        'ç': 'c', 'ä': 'a', 'ö': 'o', 'ü': 'u', "'": ''
    }
    for caracter, reemplazo in caracteres_especiales.items():
        nombre = nombre.replace(caracter, reemplazo)
    return nombre

drivers_data['Driver'] = drivers_data['Driver'].apply(limpiar_caracteres_especiales)
drivers_GP['Driver'] = drivers_GP['Driver'].apply(limpiar_caracteres_especiales)

In [ ]:
# Asegurate de que ambos 'Driver' estén en el mismo tipo (string) por si acaso
drivers_GP['Driver'] = drivers_GP['Driver'].astype(str)
drivers_data['Driver'] = drivers_data['Driver'].astype(str)

# Crear un diccionario: nombre del piloto → Driver_ID
driver_id_dict = dict(zip(drivers_data['Driver'], drivers_data['Driver_ID']))

# Asignar Driver_ID a drivers_GP según coincidencia exacta de nombre
drivers_GP['Driver_ID'] = drivers_GP['Driver'].map(driver_id_dict)



----------------------------------------------------

### Pilotos faltantes

In [ ]:
# Ordenar los pilotos faltantes alfabéticamente
faltantes = drivers_GP[drivers_GP['Driver_ID'].isna()]['Driver'].unique().tolist()
faltantes.sort()  # ordena alfabéticamente

for piloto in faltantes:
    print(piloto)

# Sumar los valores totales dentro de la variable pilotos_unicos
total = len(faltantes)
print(f'Total de pilotos únicos: {total}')

----------------------------

### Buscar pilotos por APELLIDO

In [ ]:
# Buscar pilotos en drivers_GP por el apellido en la columna 'Driver'
faltantes_apellido = drivers_GP[drivers_GP['Driver'].str.contains('Kimi', case=False)]['Driver'].unique().tolist()
faltantes_apellido.sort()  # ordena alfabéticamente 

for piloto in faltantes_apellido:
    print(piloto)

In [ ]:
# Buscar pilotos en drivers_data por el apellido en la columna 'Driver'
faltantes_apellido = drivers_data[drivers_data['Driver'].str.contains('Kimi', case=False)]['Driver'].unique().tolist()
faltantes_apellido.sort()  # ordena alfabéticamente 

for piloto in faltantes_apellido:
    print(piloto)

-------------------------

### Corregir nombres de pilotos 

In [ ]:
# Corregir nombres de pilotos en la columna 'Driver' de drivers_data
def corregir_nombres1(row):
    if row['Driver'] == 'Kimi RÃ¤ikkÃ¶nen':
        return 'Kimi Raikkonen'
    elif row['Driver'] == 'Yves Giraud Cabantous':
        return 'Yves Cabantous'
    elif row['Driver'] == 'Oscar Alfredo Galvez':
        return 'Oscar Galvez'
    elif row['Driver'] == 'Nyck De Vries':
        return 'Nyck de Vries'
    elif row['Driver'] == 'Hermano da Silva Ramos':
        return 'Hernando da Silva Ramos'
    else:
        return row['Driver']

# Corregir nombres de pilotos en la columna 'Driver' de drivers_data
def corregir_nombres2(row):
    if row['Driver'] == 'Hermano da Silva Ramos':
        return 'Hernando da Silva Ramos'
    elif row['Driver'] == 'Juan Fangio':
        return 'Juan Manuel Fangio'
    elif row['Driver'] == 'Guanyu Zhou':
        return 'Zhou Guanyu'
    else:
        return row['Driver']


In [ ]:
# Aplicar las correcciones a drivers_data
drivers_data['Driver'] = drivers_data.apply(corregir_nombres2, axis=1)
# Aplicar las correcciones a drivers_GP 
drivers_GP['Driver'] = drivers_GP.apply(corregir_nombres1, axis=1)

---------------------------

///////////////////////////////////////////////////////////

---------------------------------

In [ ]:
# Guardar los DataFrames actualizados
drivers_data.to_csv('csv_limpios/drivers_data.csv', index=False)
drivers_GP.to_csv('csv_limpios/drivers_GP.csv', index=False)

In [ ]:
# Cargar los DataFrames actualizados
drivers_data = pd.read_csv('csv_limpios/drivers_data.csv')
drivers_GP = pd.read_csv('csv_limpios/drivers_GP.csv')

In [ ]:
drivers_data

In [ ]:
drivers_GP

### IMPORTACION DEL API FASTF1 E INDAGAR EN ÉL

In [ ]:
import fastf1 as ff1
# Crear un directorio de caché para FastF1
import os
if not os.path.exists('cache'):
    os.makedirs('cache')
# Habilitrar el cache de FastF1
ff1.Cache.enable_cache('cache')

In [ ]:
# Cargar los datos de la temporada 2023
season = ff1.get_event(2023, 'Australian Grand Prix')
season25 = ff1.get_event_schedule(2025)

In [ ]:
season25